# Systolic grid across four nodes over Aurora

A 2x2 grid of 2x2 systolic blocks acting as one logical 4x4 array, one block per
node, with the wires between blocks replaced by Aurora links over the board to
board transceivers.

The blocks are not configured. Each learns its position from the first header it
receives. The only host side setting is one ComBlock register on the head node,
which selects the DMA as the source for block (0,0) instead of the Aurora link.
The host pushes the stream into the head node and reads the gathered result from
the tail node.

The compute helpers (stream build, tiling, unpack, assemble) are identical to the
single board grid and are reused unchanged. What differs is orchestration: the
four blocks live on four engines, so each accelerator call is spread across them.

Before running, the Aurora links must be up. The status cell reports this from
the design's own registers; if the transceivers are not up, the run cell will
time out, so treat that cell as a gate.

## Nodes and firmware

In [ ]:
import numpy as np
import hyperfpga_cluster as hfc
from hyperfpga_cluster import configure_logging

configure_logging()

# Reserve the four cabled nodes explicitly so the grid maps onto the physical
# 2x2 mesh. Order returned by the allocator is not the mesh order; positions are
# resolved by hostname below.
test_nodes = await hfc.request_nodes('4ge21', count=4)
for n in test_nodes:
    print(n['hostname'], n['ip'])

FIRMWARE = "sa_q4_top_wrapper_v_0_0_2"   # the Aurora bitstream (fixed dtbo name)
IP_NAME  = "sa_grid"                      # block IP uio name; confirm in probe
POS_REG  = 0                              # ComBlock register index driving the mux
POS_VAL_HEAD = 1                          # value that selects DMA on block (0,0)
POS_VAL_MESH = 0                          # value that selects Aurora elsewhere

## Physical mesh map

Fixed by the transceiver cabling of the carrier, confirmed against the exported
hardware description. Set these to the four reserved hostnames in mesh order.

In [ ]:
MESH = {
    (0, 0): "hyperfpga-4ge21-0-2",   # head: takes operands from its DMA
    (0, 1): "hyperfpga-4ge21-0-3",
    (1, 0): "hyperfpga-4ge21-1-2",
    (1, 1): "hyperfpga-4ge21-1-3",   # tail: returns the gathered C to its DMA
}
HEAD_POS, TAIL_POS = (0, 0), (1, 1)

## Constants

In [ ]:
PE, R, S, K = 2, 2, 2, 4
N, NC = R*PE, S*PE
W     = PE * 16                  # bus width in bits
APB   = W // 32                  # accumulators per beat
CBEAT = (PE*PE) // APB           # beats of one block's C tile

MAGIC = 0x53
A_FRAME, B_FRAME = 0, 1

# Two separate DMA transfers. The DMA puts TLAST at the end of a transfer, and
# load reads a frame group until TLAST, so the split is what tells a block
# where the B group ends and the A group begins.
B_BEATS    = S * (1+K)
A_BEATS    = R * (1+K)
RECV_BEATS = 1 + R*S*CBEAT       # one header, then the gathered C

# Tile shape is fixed by the bitstream. One accelerator call computes exactly
# a TM x TP tile of C from a TM x TK tile of A and a TK x TP tile of B.
TM, TP, TK = N, NC, K

# ---- the only numbers to change ----------------------------------------
M_FULL, P_FULL, K_FULL = 4, 4, 4
# ------------------------------------------------------------------------

print(f"send {B_BEATS} + {A_BEATS} beats per tile, receive {RECV_BEATS}")
print(f"tile shape {TM}x{TP}x{TK}")


## Build the stream

In [ ]:
def hdr(type_, dst_r, dst_s, ln):
    return (MAGIC << 24) | (type_ << 20) | (dst_r << 16) | (dst_s << 12) | ln

def pack(vals):
    w = 0
    for n, v in enumerate(vals):
        w |= (int(v) & 0xFFFF) << (16*n)
    return w

def build_stream(A, B):
    s = []
    for col in range(S):                       # transfer 1: B frames
        s.append(hdr(B_FRAME, 0, col, K))
        for k in range(K):
            s.append(pack(B[k, col*PE:(col+1)*PE]))
    for row in range(R):                       # transfer 2: A frames
        s.append(hdr(A_FRAME, row, 0, K))
        for k in range(K):
            s.append(pack(A[row*PE:(row+1)*PE, k]))
    return s

def unpack_C(beats):
    vals = []
    for w in beats[1:]:                        # drop the header
        for m in range(APB):
            v = (int(w) >> (32*m)) & 0xFFFFFFFF
            vals.append(v - (1 << 32) if v >> 31 else v)
    C, p = np.zeros((N, NC), dtype=np.int64), 0
    for r in range(R):
        for s_ in range(S):
            for i in range(PE):
                for j in range(PE):
                    C[r*PE+i][s_*PE+j] = vals[p]; p += 1
    return C

In [ ]:
def tile_streams(A, B):
    """Split A @ B into tile products the accelerator can run.

    Returns the per-tile streams, the (row, col) each result belongs to, and
    the tile counts. A and B are zero padded up to a whole number of tiles, so
    M_FULL, P_FULL and K_FULL do not have to be multiples of the tile shape.
    """
    m, kf = A.shape
    kf2, p = B.shape
    assert kf == kf2

    mt = -(-m // TM)
    pt = -(-p // TP)
    kt = -(-kf // TK)

    Ap = np.zeros((mt*TM, kt*TK), dtype=np.int64)
    Bp = np.zeros((kt*TK, pt*TP), dtype=np.int64)
    Ap[:m, :kf] = A
    Bp[:kf, :p] = B

    streams, index = [], []
    for i in range(mt):
        for j in range(pt):
            for k in range(kt):
                streams.append(build_stream(Ap[i*TM:(i+1)*TM, k*TK:(k+1)*TK],
                                            Bp[k*TK:(k+1)*TK, j*TP:(j+1)*TP]))
                index.append((i, j))
    return streams, index, (mt, pt, kt)


def assemble_C(all_beats, index, shape):
    """Accumulate the tile results into the full C matrix."""
    m, p = shape
    mt = -(-m // TM)
    pt = -(-p // TP)
    C = np.zeros((mt*TM, pt*TP), dtype=np.int64)
    for beats, (i, j) in zip(all_beats, index):
        C[i*TM:(i+1)*TM, j*TP:(j+1)*TP] += unpack_C(beats)
    return C[:m, :p]


## Test data

In [ ]:
AMAX = 1000
assert TK * AMAX * AMAX < 2**31          # per tile, the accumulator is 32 bit

rng = np.random.default_rng(0)
A = rng.integers(-AMAX, AMAX+1, size=(M_FULL, K_FULL)).astype(np.int64)
B = rng.integers(-AMAX, AMAX+1, size=(K_FULL, P_FULL)).astype(np.int64)
C_golden = A @ B

streams, index, (mt, pt, kt) = tile_streams(A, B)
n_tiles = len(streams)
in_beats = B_BEATS + A_BEATS

print(f"{M_FULL}x{K_FULL} @ {K_FULL}x{P_FULL}")
print(f"{mt} x {pt} x {kt} tiles = {n_tiles} accelerator calls")
print(f"buffer needed = {n_tiles*(in_beats+RECV_BEATS)*(W//8)} bytes")


## Start engines and probe

One engine per node. The probe lists the uio devices, ComBlocks and DMAs on each
node, so IP_NAME and the mesh mapping can be confirmed against real names before
anything is programmed against them.

In [ ]:
def probe():
    import os, socket
    import hyperfpga_comutils as hf
    hw = hf.HardwareManager.detect()
    uio = []
    for dev in sorted(os.listdir("/sys/class/uio")):
        try:
            uio.append((dev, open(f"/sys/class/uio/{dev}/name").read().strip()))
        except Exception:
            pass
    return {"host": socket.gethostname(), "uio": uio,
            "comblocks": {k: repr(v) for k, v in hw.get("comblocks", {}).items()},
            "dmas": {k: repr(v) for k, v in hw.get("dmas", {}).items()}}


PROGRAM = True         # set False once the firmware is already loaded

cluster = (hfc.HyperFPGACluster(nodes=test_nodes, firmware=FIRMWARE)
           if PROGRAM else hfc.HyperFPGACluster(nodes=test_nodes))
rc = None
await cluster.configure()
cluster.create_profile(mpi=False)
rc = await cluster.start_and_connect()
rc.wait_for_engines(n=len(test_nodes), timeout=120)

info = {i: rc[i].apply_sync(probe) for i in rc.ids}
for i, p in info.items():
    print(f"engine {i}  {p['host']}")
    print("   uio      ", p["uio"])
    print("   comblocks", p["comblocks"])
    print("   dmas     ", p["dmas"])

## Resolve engines to mesh positions

Maps each grid position to the engine on the matching host. HEAD and TAIL, and
the middle engines, follow from that.

In [ ]:
host_to_engine = {p["host"]: i for i, p in info.items()}
pos_to_engine  = {pos: host_to_engine[h] for pos, h in MESH.items()}
engine_to_pos  = {e: pos for pos, e in pos_to_engine.items()}

HEAD = pos_to_engine[HEAD_POS]
TAIL = pos_to_engine[TAIL_POS]
MID  = [e for e in rc.ids if e not in (HEAD, TAIL)]

for pos in sorted(pos_to_engine):
    e = pos_to_engine[pos]
    tag = "HEAD" if e == HEAD else ("TAIL" if e == TAIL else "mid")
    print(pos, "->", info[e]["host"], f"engine {e} [{tag}]")

## Aurora status (gate)

Reads the design's status registers on every node and reports link state per
transceiver, plus the three on chip frequency counters. The map below matches
the register file in the bitstream.

Run this before any tile. Every used link must read channel up. If they read
down with USER_CLK at zero, the MGT reference clock is not present and the run
cell will time out; stop here and resolve the clock first.

In [ ]:
FIELDS = ["channel_up","crc_pass_fail_n","crc_valid","frame_err","hard_err",
          "lane_up","pll_not_locked","rx_resetdone","soft_err","tx_lock",
          "tx_resetdone","link_reset","sys_reset","sync_clk","gt_powergood"]
GTH = [("A up-RX", 0), ("B left-RX", 15), ("C down-TX", 30), ("D right-TX", 45)]
STATIC_BASE = 32                          # word offset of static_in[0]

def aurora_status():
    import os, mmap
    dev = None
    for d in sorted(os.listdir("/sys/class/uio")):
        try:
            if open(f"/sys/class/uio/{d}/name").read().strip().startswith("axi"):
                if "static_regs" in open(f"/sys/class/uio/{d}/name").read():
                    dev = d
        except Exception:
            pass
    # fall back to explicit name match
    if dev is None:
        for d in sorted(os.listdir("/sys/class/uio")):
            try:
                if "static_regs" in open(f"/sys/class/uio/{d}/name").read():
                    dev = d; break
            except Exception:
                pass
    if dev is None:
        return "static regs device not found"
    f = os.open(f"/dev/{dev}", os.O_RDWR | os.O_SYNC)
    m = mmap.mmap(f, 0x1000)
    w = [int.from_bytes(m[i*4:i*4+4], "little") for i in range(124)]
    m.close(); os.close(f)
    return w

def show_status():
    all_up = True
    for i in rc.ids:
        w = rc[i].apply_sync(aurora_status)
        pos = engine_to_pos[i]
        print(f"--- engine {i} {pos} {info[i]['host']}")
        if isinstance(w, str):
            print("   ", w); all_up = False; continue
        for name, base in GTH:
            v = {FIELDS[k]: (1 if w[STATIC_BASE+base+k] else 0) for k in range(15)}
            up = "UP" if v["channel_up"] else "down"
            if not v["channel_up"]:
                all_up = False
            print(f"   GTH {name}: channel={up} lane={v['lane_up']} "
                  f"pll_locked={1-v['pll_not_locked']} txlock={v['tx_lock']} "
                  f"hard={v['hard_err']} pgood={v['gt_powergood']}")
        print(f"   freq: INIT={w[121]}  SYS={w[122]}  USER={w[123]}")
    print()
    print("ALL CHANNELS UP" if all_up else "LINKS NOT UP - do not run tiles yet")
    return all_up

links_up = show_status()

## Per node setup

Runs once on every engine. Finds the block, the DMA and the ComBlock, writes the
position register, allocates a DMA buffer, and keeps the native handles in the
engine namespace, since they cannot be shipped back.

In [ ]:
def setup_node(ip_name, is_head, pos_reg, val_head, val_mesh):
    import os
    import __main__ as M
    import hyperfpga_comutils as hf

    hw = hf.HardwareManager.detect()
    devs = []
    for dev in sorted(os.listdir("/sys/class/uio")):
        try:
            if open(f"/sys/class/uio/{dev}/name").read().strip() == ip_name:
                devs.append(dev)
        except Exception:
            pass
    if len(devs) != 1:
        raise RuntimeError(f"expected one '{ip_name}', found {devs}")

    cb = list(hw.get("comblocks", {}).values())[0]
    cb.write_reg(pos_reg, val_head if is_head else val_mesh)

    M.NODE = {"blk": hf.HlsControl(f"/dev/{devs[0]}"),
              "dma": list(hw["dmas"].values())[0],
              "cb":  cb,
              "buf": hf.UDMABuffer(0, format_in="unsigned32",
                                   format_out="unsigned32", signed=False)}
    return devs[0], M.NODE["buf"].size


for i in rc.ids:
    is_head = (i == HEAD)
    r = rc[i].apply_sync(setup_node, IP_NAME, is_head, POS_REG,
                         POS_VAL_HEAD, POS_VAL_MESH)
    print(i, engine_to_pos[i], info[i]["host"],
          "HEAD" if is_head else "", r)

## Run

One tile per round trip, the same order as the single board grid, spread over the
engines. The receive channel is armed on every node, not only the tail, because
the downward output of each block passes through a stream broadcaster whose
second output feeds the local DMA; if a node's S2MM is not armed, that broadcaster
stalls and the whole grid halts. The head is started last and then pushes B then
A from its DMA.

The two returned times are wall clock on the head (push) and the tail (wait).
They include ipyparallel round trips and are not the fabric latency; they only
confirm the transfer completed. A clean fabric latency figure needs the on chip
ILA on the transceivers, which is a separate measurement.

In [ ]:
def arm_recv(recv_beats, wbytes):
    import __main__ as M
    n = M.NODE
    # generous receive size so a non-tail node absorbs pass-through traffic
    n["dma"].start_s2mm(n["buf"].phys_addr, recv_beats * wbytes)
    n["blk"].start()

def start_block():
    import __main__ as M
    M.NODE["blk"].start()

def push_head(stream, b_beats, a_beats, wbytes):
    import time
    import __main__ as M
    n = M.NODE
    buf, dma = n["buf"], n["dma"]
    buf.write_values([int(x) for x in stream], 0)
    buf.sync_for_device(0, (b_beats + a_beats) * wbytes)
    n["blk"].start()
    t0 = time.perf_counter()
    dma.start_mm2s(buf.phys_addr, b_beats * wbytes)
    dma.wait_mm2s(5.0)
    dma.start_mm2s(buf.phys_addr + b_beats * wbytes, a_beats * wbytes)
    dma.wait_mm2s(5.0)
    return time.perf_counter() - t0

def read_tail(recv_beats, wbytes, timeout=5.0):
    import time
    import __main__ as M
    n = M.NODE
    t0 = time.perf_counter()
    n["dma"].wait_s2mm(timeout)
    dt = time.perf_counter() - t0
    n["buf"].sync_for_cpu(0, recv_beats * wbytes)
    return [int(x) for x in n["buf"].read_values(recv_beats, 0)], dt


def run_tile(stream):
    wbytes = W // 8
    # arm receive on every node; tail gets the real C size, others a safe buffer
    for i in MID + [TAIL]:
        rc[i].apply_sync(arm_recv, RECV_BEATS, wbytes)
    t_push = rc[HEAD].apply_sync(push_head, stream, B_BEATS, A_BEATS, wbytes)
    beats, t_wait = rc[TAIL].apply_sync(read_tail, RECV_BEATS, wbytes)
    return beats, t_push, t_wait


if not links_up:
    raise RuntimeError("Aurora links are not up; see the status cell above")

all_beats, times = [], []
for k, s in enumerate(streams):
    b, tp, tw = run_tile(s)
    all_beats.append(b)
    times.append((tp, tw))
    print(f"tile {k}: push {tp*1e6:8.1f} us   tail wait {tw*1e6:8.1f} us")

C_hw = assemble_C(all_beats, index, (M_FULL, P_FULL))

## Check

In [ ]:
h = all_beats[0][0]
print(f"header magic 0x{(h >> 24) & 0xFF:02X}  (expect 0x{MAGIC:02X})")

err = np.abs(C_hw - C_golden)
print(f"max error = {err.max()}")
print("PASS" if err.max() == 0 else "FAIL")

## Release

In [ ]:
if rc is not None:
    rc.shutdown()
    await cluster.stop_cluster()
await cluster.clean_cluster()
await cluster.release_cluster()
print("released")